In [ ]:
import sys
sys.path.insert(0, "..")  # add project root to path so we can import src

from src.config import (
    PROJECT_ROOT,
    SALES_CLEAN_PATH,
    CASHFLOW_ACCRUAL_PATH,
    BANKING_DELAY_DAYS,
    SIZE_PERCENTILE,
)

print(f"Project root: {PROJECT_ROOT}")
print(f"Sales data path: {SALES_CLEAN_PATH}")
print(f"Cashflow accrual path: {CASHFLOW_ACCRUAL_PATH}")
print(f"Banking delay: {BANKING_DELAY_DAYS} days")
print(f"Size threshold percentile: {SIZE_PERCENTILE}")

# Verify paths actually point to real files
print(f"\nSales file exists: {SALES_CLEAN_PATH.exists()}")
print(f"Accrual target exists: {CASHFLOW_ACCRUAL_PATH.exists()}")

In [ ]:
# Test the data_loader module
from src.data_loader import load_invoices, load_targets

# Load everything
invoices = load_invoices()
targets = load_targets()

# Verify shapes match Phase 2 outputs
print("Invoices:")
print(f"  Sales:     {invoices['sales'].shape}")
print(f"  Purchases: {invoices['purchases'].shape}")

print("\nTargets:")
print(f"  Accrual:   {targets['accrual'].shape}")
print(f"  Projected: {targets['projected'].shape}")

print(f"\nAccrual columns:   {list(targets['accrual'].columns)}")
print(f"Projected columns: {list(targets['projected'].columns)}")

print(f"\nAccrual target — deficit weeks: "
      f"{(targets['accrual']['net_cashflow'] < 0).sum()} of {len(targets['accrual'])} "
      f"({100 * (targets['accrual']['net_cashflow'] < 0).mean():.1f}%)")
print(f"Projected target — deficit weeks: "
      f"{(targets['projected']['net_cashflow'] < 0).sum()} of {len(targets['projected'])} "
      f"({100 * (targets['projected']['net_cashflow'] < 0).mean():.1f}%)")

In [ ]:
# Test the targets module by regenerating both targets from scratch
import pandas as pd
import json
from src.data_loader import load_invoices, load_targets
from src.targets import (
    build_accrual_target,
    build_payment_projected_target,
    compute_per_customer_thresholds,
)
from src.config import TERMS_FILE_PATH, GSTIN_MAPPING_PATH

# Load raw invoice data
invoices = load_invoices()
sales = invoices['sales']
purchases = invoices['purchases']

# Load terms file and apply GSTIN pseudonymisation
buyer_terms = pd.read_excel(TERMS_FILE_PATH, sheet_name='Customers (Sales)')
supplier_terms = pd.read_excel(TERMS_FILE_PATH, sheet_name='Suppliers (Purchases)')

with open(GSTIN_MAPPING_PATH, 'r') as f:
    gstin_map = json.load(f)

# Clean whitespace and map to pseudonymised GSTINs
for df in (buyer_terms, supplier_terms):
    df['Payment Terms (days)'] = df['Payment Terms (days)'].astype(str).str.strip()
    df['GSTIN_pseudo'] = df['GSTIN'].map(gstin_map)
    df.drop(columns=['GSTIN'], inplace=True)
    df.rename(columns={'GSTIN_pseudo': 'GSTIN'}, inplace=True)

# Deduplicate on GSTIN
def dedupe_terms(df):
    return (df.dropna(subset=['GSTIN'])
              .sort_values('Number of Invoices', ascending=False)
              .drop_duplicates(subset='GSTIN', keep='first')
              .reset_index(drop=True))

buyer_terms_dedup = dedupe_terms(buyer_terms)
supplier_terms_dedup = dedupe_terms(supplier_terms)

# Regenerate both targets from scratch
accrual_new = build_accrual_target(sales, purchases)
projected_new = build_payment_projected_target(
    sales, purchases, buyer_terms_dedup, supplier_terms_dedup,
)

# Compare against the saved Phase 2 targets
saved = load_targets()

print("Accrual target:")
print(f"  Regenerated: {accrual_new.shape},  deficit weeks: "
      f"{(accrual_new['net_cashflow'] < 0).sum()} "
      f"({100 * (accrual_new['net_cashflow'] < 0).mean():.1f}%)")
print(f"  Saved:       {saved['accrual'].shape},  deficit weeks: "
      f"{(saved['accrual']['net_cashflow'] < 0).sum()} "
      f"({100 * (saved['accrual']['net_cashflow'] < 0).mean():.1f}%)")

print("\nPayment-projected target:")
print(f"  Regenerated: {projected_new.shape},  deficit weeks: "
      f"{(projected_new['net_cashflow'] < 0).sum()} "
      f"({100 * (projected_new['net_cashflow'] < 0).mean():.1f}%)")
print(f"  Saved:       {saved['projected'].shape},  deficit weeks: "
      f"{(saved['projected']['net_cashflow'] < 0).sum()} "
      f"({100 * (saved['projected']['net_cashflow'] < 0).mean():.1f}%)")

# Numeric equality check (values should match to within floating point)
accrual_diff = (accrual_new['net_cashflow'] - saved['accrual']['net_cashflow']).abs().max()
projected_diff = (projected_new['net_cashflow'] - saved['projected']['net_cashflow']).abs().max()

print(f"\nMax absolute difference (regenerated vs saved):")
print(f"  Accrual:   {accrual_diff:,.2f}")
print(f"  Projected: {projected_diff:,.2f}")

In [ ]:
# Diagnose the payment-projected mismatch
# Compare regenerated vs saved on a week-by-week basis to find the divergent weeks

comparison = pd.DataFrame({
    'saved_net': saved['projected']['net_cashflow'],
    'regen_net': projected_new['net_cashflow'],
})
comparison['diff'] = comparison['regen_net'] - comparison['saved_net']
comparison['abs_diff'] = comparison['diff'].abs()

# Show weeks where they differ
divergent = comparison[comparison['abs_diff'] > 1.0].sort_values('abs_diff', ascending=False)

print(f"Number of divergent weeks: {len(divergent)}")
print(f"\nTop 10 divergent weeks:")
print(divergent.head(10))

# Show if the regenerated series is missing any weeks the saved has
saved_only_weeks = set(saved['projected'].index) - set(projected_new.index)
regen_only_weeks = set(projected_new.index) - set(saved['projected'].index)
print(f"\nWeeks in saved but not regenerated: {len(saved_only_weeks)}")
if saved_only_weeks:
    print(f"  Dates: {sorted(saved_only_weeks)}")
print(f"Weeks in regenerated but not saved: {len(regen_only_weeks)}")
if regen_only_weeks:
    print(f"  Dates: {sorted(regen_only_weeks)}")

# Also check the total volumes
print(f"\nTotal sales (regenerated): {projected_new['sales'].sum():,.0f}")
print(f"Total sales (saved):       {saved['projected']['sales'].sum():,.0f}")
print(f"Total purchases (regen):   {projected_new['purchases'].sum():,.0f}")
print(f"Total purchases (saved):   {saved['projected']['purchases'].sum():,.0f}")

In [ ]:
# Overwrite the saved payment-projected target with the new methodology
from src.config import CASHFLOW_PROJECTED_PATH

projected_new.to_parquet(CASHFLOW_PROJECTED_PATH)
print(f"Overwrote {CASHFLOW_PROJECTED_PATH.name} with per-buyer + fixed-₹12K methodology")
print(f"New shape: {projected_new.shape}")
print(f"New deficit weeks: {(projected_new['net_cashflow'] < 0).sum()} "
      f"({100 * (projected_new['net_cashflow'] < 0).mean():.1f}%)")

In [ ]:
# Force reload of the modules we care about
import importlib
import src.config
import src.targets

importlib.reload(src.config)
importlib.reload(src.targets)

# Re-import the specific names we use
from src.config import PURCHASE_SIZE_THRESHOLD, CASHFLOW_PROJECTED_PATH
from src.targets import (
    build_accrual_target,
    build_payment_projected_target,
    compute_per_customer_thresholds,
)

print(f"PURCHASE_SIZE_THRESHOLD is now: {PURCHASE_SIZE_THRESHOLD}")

In [ ]:
projected_new.to_parquet(CASHFLOW_PROJECTED_PATH)
print(f"Overwrote {CASHFLOW_PROJECTED_PATH.name}")
print(f"Correct methodology now saved to disk.")
print(f"  Shape: {projected_new.shape}")
print(f"  Deficit weeks: {(projected_new['net_cashflow'] < 0).sum()} "
      f"({100 * (projected_new['net_cashflow'] < 0).mean():.1f}%)")

In [ ]:
from src.data_loader import load_targets

# Re-import to bypass any caching
import importlib
import src.data_loader
importlib.reload(src.data_loader)
from src.data_loader import load_targets

verified = load_targets()
print(f"Reloaded from disk:")
print(f"  Projected shape: {verified['projected'].shape}")
print(f"  Deficit weeks: {(verified['projected']['net_cashflow'] < 0).sum()} "
      f"({100 * (verified['projected']['net_cashflow'] < 0).mean():.1f}%)")

In [ ]:
# Test the features module by building features from the accrual target
import importlib
import src.features
importlib.reload(src.features)
from src.features import (
    build_calendar_features,
    build_lag_features,
    build_rolling_features,
    build_payment_cycle_features,
    build_all_features,
)
from src.config import BASE_SERIES
from src.data_loader import load_targets

targets = load_targets()
accrual = targets['accrual']

features = build_all_features(accrual)

print(f"Feature DataFrame shape: {features.shape}")
print(f"Total columns:           {len(features.columns)}")
print(f"Expected:                72 columns")
print()

base_cols = [c for c in features.columns if c in BASE_SERIES]
calendar_cols = ['year', 'quarter', 'month', 'week_of_year', 'week_of_month', 'is_quarter_end']
lag_cols = [c for c in features.columns if '_lag_' in c]
rolling_cols = [c for c in features.columns if '_roll' in c]
paycycle_cols = [c for c in features.columns 
                  if 'paywindow' in c or 'expected' in c or 'ratio' in c]

print(f"Base series:      {len(base_cols)} columns")
print(f"Calendar:         {len(calendar_cols)} columns")
print(f"Lag features:     {len(lag_cols)} columns")
print(f"Rolling features: {len(rolling_cols)} columns")
print(f"Payment-cycle:    {len(paycycle_cols)} columns")

total = len(base_cols) + len(calendar_cols) + len(lag_cols) + len(rolling_cols) + len(paycycle_cols)
print(f"Sum:              {total} columns")

first_row = features.iloc[0]
rolling_nans = first_row[[c for c in features.columns if '_roll' in c]].isna().sum()
lag_nans = first_row[[c for c in features.columns if '_lag_' in c]].isna().sum()
print(f"\nLeakage check — first row NaN counts:")
print(f"  Rolling features:  {rolling_nans}/{len(rolling_cols)}  (should equal {len(rolling_cols)})")
print(f"  Lag features:      {lag_nans}/{len(lag_cols)}  (should equal {len(lag_cols)})")

In [ ]:
# Test the validation module
import importlib
import numpy as np
import src.validation
importlib.reload(src.validation)
from src.validation import walk_forward_splits, compute_mase

# Test walk-forward splits on the accrual target (158 weeks)
from src.data_loader import load_targets
targets = load_targets()
accrual = targets['accrual']

print(f"Accrual target: {len(accrual)} weeks")
print(f"\nWalk-forward splits (5 folds, min_train=52):")
print(f"{'Fold':<6}{'Train range':<20}{'Test range':<20}{'Train size':>12}{'Test size':>12}")

for fold_idx, (train_idx, test_idx) in enumerate(walk_forward_splits(accrual)):
    train_range = f"[{train_idx[0]}-{train_idx[-1]}]"
    test_range = f"[{test_idx[0]}-{test_idx[-1]}]"
    print(f"{fold_idx+1:<6}{train_range:<20}{test_range:<20}"
          f"{len(train_idx):>12}{len(test_idx):>12}")

# Test MASE on a trivial example
print(f"\n--- MASE tests ---")

# Perfect prediction: MASE = 0
y_train = np.array([100, 110, 105, 120, 115])
y_true = np.array([125, 130])
y_pred_perfect = np.array([125, 130])
print(f"Perfect prediction MASE:      {compute_mase(y_true, y_pred_perfect, y_train):.3f}  (expected 0.000)")

# Naive prediction (last training value carried forward): MASE ~ 1
y_pred_naive = np.array([115, 115])
naive_mase = compute_mase(y_true, y_pred_naive, y_train)
print(f"Naive prediction MASE:        {naive_mase:.3f}  (expected around 1)")

# Way off prediction: high MASE
y_pred_bad = np.array([0, 0])
bad_mase = compute_mase(y_true, y_pred_bad, y_train)
print(f"Bad prediction (zeros) MASE:  {bad_mase:.3f}  (expected much higher than 1)")

In [ ]:
# Test the baselines module — full walk-forward evaluation of naive seasonal and ARIMA
import importlib
import src.models.baselines
importlib.reload(src.models.baselines)
from src.models.baselines import fit_predict_naive_seasonal, fit_predict_arima

from src.data_loader import load_targets
from src.validation import walk_forward_splits, compute_mase

# Run on accrual target
accrual = load_targets()['accrual']
y = accrual['net_cashflow'].values

# Collect MASE scores per fold for each model
naive_mases = []
arima_mases = []

print(f"{'Fold':<6}{'Naive MASE':>15}{'ARIMA MASE':>15}")
print("-" * 36)

for fold_idx, (train_idx, test_idx) in enumerate(walk_forward_splits(accrual)):
    y_train = y[train_idx]
    y_test = y[test_idx]
    horizon = len(test_idx)

    # Naive seasonal
    y_pred_naive = fit_predict_naive_seasonal(y_train, horizon)
    naive_mase = compute_mase(y_test, y_pred_naive, y_train)
    naive_mases.append(naive_mase)

    # ARIMA with auto order selection
    try:
        y_pred_arima = fit_predict_arima(y_train, horizon, order=(1, 1, 1))
        arima_mase = compute_mase(y_test, y_pred_arima, y_train)
        arima_mases.append(arima_mase)
    except Exception as e:
        arima_mase = float('nan')
        arima_mases.append(arima_mase)
        print(f"  Fold {fold_idx+1}: ARIMA failed with {type(e).__name__}: {e}")

    print(f"{fold_idx+1:<6}{naive_mase:>15.3f}{arima_mase:>15.3f}")

print("-" * 36)
print(f"{'Mean':<6}{np.mean(naive_mases):>15.3f}{np.nanmean(arima_mases):>15.3f}")

In [ ]:
import sys
print(f"Python executable: {sys.executable}")
print(f"Python version:    {sys.version}")

# Check if pmdarima is importable from THIS Python
try:
    import pmdarima
    print(f"pmdarima version:  {pmdarima.__version__}")
    print(f"pmdarima location: {pmdarima.__file__}")
except ImportError as e:
    print(f"pmdarima import failed: {e}")

# Check statsmodels
try:
    import statsmodels
    print(f"statsmodels version:  {statsmodels.__version__}")
except ImportError:
    print("statsmodels not found")
    

In [ ]:
import sys
print(f"Python executable: {sys.executable}")

try:
    import pmdarima
    print(f"pmdarima version:  {pmdarima.__version__}")
    print(f"pmdarima location: {pmdarima.__file__}")
except ImportError as e:
    print(f"pmdarima import failed: {e}")

In [ ]:
import sys
print(f"Python executable: {sys.executable}")

try:
    import xgboost
    print(f"xgboost {xgboost.__version__} — location: {xgboost.__file__}")
except ImportError as e:
    print(f"xgboost import FAILED: {e}")

try:
    import lightgbm
    print(f"lightgbm {lightgbm.__version__} — location: {lightgbm.__file__}")
except ImportError as e:
    print(f"lightgbm import FAILED: {e}")

In [ ]:
import sys
!{sys.executable} -m pip install xgboost lightgbm

In [ ]:
# Run XGBoost and LightGBM against both targets, compare with ARIMA baseline
import importlib
import src.models.tree_models
import src.models.evaluation
importlib.reload(src.models.tree_models)
importlib.reload(src.models.evaluation)

from src.models.tree_models import fit_predict_xgboost, fit_predict_lightgbm
from src.models.evaluation import run_walk_forward_evaluation
from src.data_loader import load_targets
from src.features import build_all_features

# Load both targets and build features for each
targets = load_targets()

results = {}

for target_name, target_df in targets.items():
    features = build_all_features(target_df)
    print(f"\n=== Target: {target_name.upper()} ({len(target_df)} weeks) ===")

    for model_name, model_fn in [('XGBoost', fit_predict_xgboost),
                                  ('LightGBM', fit_predict_lightgbm)]:
        eval_result = run_walk_forward_evaluation(model_fn, features)
        results[f'{target_name}_{model_name}'] = eval_result
        print(f"  {model_name:10s}  "
              f"mean MASE: {eval_result['mean_mase']:.3f}  "
              f"std: {eval_result['std_mase']:.3f}  "
              f"per-fold: {[f'{x:.3f}' for x in eval_result['mase_per_fold']]}")

# Baseline reference
print(f"\n=== BASELINE for comparison ===")
print(f"  ARIMA(1,1,1) on accrual: mean MASE 0.644 (from earlier smoke test)")

In [ ]:
# Check whether sales and purchases are being used as features
from src.features import build_all_features
from src.data_loader import load_targets

target = load_targets()['accrual']
features = build_all_features(target)

# Show what features would go into the model (X after dropping target)
X_columns = [c for c in features.columns if c != 'net_cashflow']
print(f"Feature columns fed to model: {len(X_columns)}")
print(f"\nBase series in features:")
for col in ['sales', 'purchases', 'net_cashflow']:
    if col in X_columns:
        print(f"  {col:15s} — IN X (leakage risk if current-week)")
    else:
        print(f"  {col:15s} — dropped, not in X")

In [ ]:
# Investigate the XGBoost fold-3 outlier on the projected target
from src.models.tree_models import fit_predict_xgboost
from src.models.evaluation import run_walk_forward_evaluation

target = load_targets()['projected']
features = build_all_features(target).dropna()
print(f"Clean features (post-NaN-drop): {features.shape}")

# Look at what fold 3 actually contains
from src.validation import walk_forward_splits
for fold_idx, (train_pos, test_pos) in enumerate(walk_forward_splits(features)):
    if fold_idx == 2:  # 0-indexed fold 3
        train = features.iloc[train_pos]
        test = features.iloc[test_pos]
        print(f"\nFold 3:")
        print(f"  Train size: {len(train)}, Test size: {len(test)}")
        print(f"  Train net_cashflow range: [{train['net_cashflow'].min():,.0f}, {train['net_cashflow'].max():,.0f}]")
        print(f"  Test net_cashflow range:  [{test['net_cashflow'].min():,.0f}, {test['net_cashflow'].max():,.0f}]")
        print(f"  Train mean: {train['net_cashflow'].mean():,.0f}")
        print(f"  Test mean:  {test['net_cashflow'].mean():,.0f}")

In [ ]:
# Investigate the XGBoost fold-3 outlier on the projected target
from src.models.tree_models import fit_predict_xgboost
from src.models.evaluation import run_walk_forward_evaluation

target = load_targets()['projected']
features = build_all_features(target).dropna()
print(f"Clean features (post-NaN-drop): {features.shape}")

# Look at what fold 3 actually contains
from src.validation import walk_forward_splits
for fold_idx, (train_pos, test_pos) in enumerate(walk_forward_splits(features)):
    if fold_idx == 2:  # 0-indexed fold 3
        train = features.iloc[train_pos]
        test = features.iloc[test_pos]
        print(f"\nFold 3:")
        print(f"  Train size: {len(train)}, Test size: {len(test)}")
        print(f"  Train net_cashflow range: [{train['net_cashflow'].min():,.0f}, {train['net_cashflow'].max():,.0f}]")
        print(f"  Test net_cashflow range:  [{test['net_cashflow'].min():,.0f}, {test['net_cashflow'].max():,.0f}]")
        print(f"  Train mean: {train['net_cashflow'].mean():,.0f}")
        print(f"  Test mean:  {test['net_cashflow'].mean():,.0f}")

In [ ]:
# Test ARIMA on both targets for fair comparison
from src.models.baselines import fit_predict_arima
from src.validation import walk_forward_splits, compute_mase
import numpy as np

for target_name, target_df in load_targets().items():
    y = target_df['net_cashflow'].values
    mases = []
    for train_idx, test_idx in walk_forward_splits(target_df):
        y_train = y[train_idx]
        y_test = y[test_idx]
        try:
            y_pred = fit_predict_arima(y_train, len(test_idx), order=(1,1,1))
            mases.append(compute_mase(y_test, y_pred, y_train))
        except Exception as e:
            print(f"  ARIMA failed on fold: {e}")
            mases.append(float('nan'))
    print(f"ARIMA on {target_name}: mean MASE = {np.nanmean(mases):.3f}, per-fold: {[f'{x:.3f}' for x in mases]}")

In [ ]:
# Save baseline results table for use in Chapter 4
import pandas as pd
from src.config import PROCESSED_DATA_DIR
import os

baseline_results = pd.DataFrame({
    'Model': ['ARIMA(1,1,1)', 'XGBoost', 'LightGBM',
              'ARIMA(1,1,1)', 'XGBoost', 'LightGBM'],
    'Target': ['Accrual', 'Accrual', 'Accrual',
               'Projected', 'Projected', 'Projected'],
    'Mean_MASE': [0.644, 1.051, 0.886, 0.569, 1.176, 0.806],
    'Std_MASE': [np.std([0.678, 0.633, 0.604, 0.451, 0.853]),
                 0.346, 0.241,
                 np.std([0.477, 0.834, 0.494, 0.561, 0.480]),
                 0.729, 0.104],
})

os.makedirs(f"{PROCESSED_DATA_DIR}/results", exist_ok=True)
baseline_results.to_csv(f"{PROCESSED_DATA_DIR}/results/baseline_no_augmentation.csv", index=False)

print("Saved baseline results to data/processed/results/baseline_no_augmentation.csv")
print("\nFinal Phase 3 baseline table (no augmentation):")
print(baseline_results.to_string(index=False))

In [ ]:
# Extract the exact data statistics needed for Chapter 3.1
from src.data_loader import load_invoices, load_targets
import pandas as pd

inv = load_invoices()
sales = inv['sales']
purchases = inv['purchases']
targets = load_targets()

print("=" * 60)
print("Chapter 3.1 statistics — extracted from saved data")
print("=" * 60)

# Invoice counts
print(f"\nSales invoices:    {len(sales):,}")
print(f"Purchase invoices: {len(purchases):,}")

# Date ranges
print(f"\nSales date range:    {sales['Date'].min().date()} to {sales['Date'].max().date()}")
print(f"Purchase date range: {purchases['Date'].min().date()} to {purchases['Date'].max().date()}")

# Time span in weeks
accrual_weeks = len(targets['accrual'])
projected_weeks = len(targets['projected'])
print(f"\nAccrual target span:   {accrual_weeks} weeks")
print(f"Projected target span: {projected_weeks} weeks")

# Sales invoice distribution
sales_amt = sales['Gross Total']
print(f"\nSales invoice amounts (INR):")
print(f"  Median:  {sales_amt.median():>15,.2f}")
print(f"  Mean:    {sales_amt.mean():>15,.2f}")
print(f"  Std:     {sales_amt.std():>15,.2f}")
print(f"  Min:     {sales_amt.min():>15,.2f}")
print(f"  Max:     {sales_amt.max():>15,.2f}")
print(f"  75th %:  {sales_amt.quantile(0.75):>15,.2f}")
print(f"  95th %:  {sales_amt.quantile(0.95):>15,.2f}")

# Purchase invoice distribution
purch_amt = purchases['Gross Total']
print(f"\nPurchase invoice amounts (INR):")
print(f"  Median:  {purch_amt.median():>15,.2f}")
print(f"  Mean:    {purch_amt.mean():>15,.2f}")
print(f"  Std:     {purch_amt.std():>15,.2f}")
print(f"  Min:     {purch_amt.min():>15,.2f}")
print(f"  Max:     {purch_amt.max():>15,.2f}")
print(f"  75th %:  {purch_amt.quantile(0.75):>15,.2f}")
print(f"  95th %:  {purch_amt.quantile(0.95):>15,.2f}")

# Unique counterparties
n_buyers = sales['GSTIN/UIN'].nunique()
n_suppliers = purchases['GSTIN/UIN'].nunique()
print(f"\nUnique buyers (by pseudonymised GSTIN):    {n_buyers}")
print(f"Unique suppliers (by pseudonymised GSTIN): {n_suppliers}")

# GSTIN completeness
sales_gstin_missing = sales['GSTIN/UIN'].isnull().sum()
purch_gstin_missing = purchases['GSTIN/UIN'].isnull().sum()
print(f"\nSales invoices with missing GSTIN:    {sales_gstin_missing} ({100*sales_gstin_missing/len(sales):.1f}%)")
print(f"Purchase invoices with missing GSTIN: {purch_gstin_missing} ({100*purch_gstin_missing/len(purchases):.1f}%)")

# Total transactional volume
print(f"\nTotal sales volume (INR):     {sales_amt.sum():,.0f}")
print(f"Total purchase volume (INR):  {purch_amt.sum():,.0f}")

# Top 5 customers concentration
top5_sales = (sales.groupby('GSTIN/UIN')['Gross Total'].sum()
              .sort_values(ascending=False).head(5).sum())
print(f"\nRevenue concentration:")
print(f"  Top 5 buyers total:     {top5_sales:,.0f} ({100*top5_sales/sales_amt.sum():.1f}% of total sales)")

In [ ]:
import sys
!{sys.executable} -m pip install prophet

In [ ]:
try:
    from prophet import Prophet
    import prophet
    print(f"Prophet installed successfully, version {prophet.__version__}")
except ImportError as e:
    print(f"Prophet import failed: {e}")

In [ ]:
# Test Prophet on both targets
import importlib
import src.models.prophet_model
importlib.reload(src.models.prophet_model)
from src.models.prophet_model import fit_predict_prophet

from src.models.evaluation import run_walk_forward_evaluation
from src.data_loader import load_targets
from src.features import build_all_features

for target_name, target_df in load_targets().items():
    features = build_all_features(target_df)
    result = run_walk_forward_evaluation(fit_predict_prophet, features)
    print(f"Prophet on {target_name}:  mean MASE = {result['mean_mase']:.3f}  "
          f"std = {result['std_mase']:.3f}  "
          f"per-fold: {[f'{x:.3f}' for x in result['mase_per_fold']]}")

In [ ]:
# Update baseline results table with Prophet
import pandas as pd
import numpy as np
from src.config import PROCESSED_DATA_DIR
import os

baseline_results = pd.DataFrame({
    'Model':     ['ARIMA(1,1,1)', 'XGBoost',  'LightGBM', 'Prophet',
                  'ARIMA(1,1,1)', 'XGBoost',  'LightGBM', 'Prophet'],
    'Target':    ['Accrual',      'Accrual',  'Accrual',  'Accrual',
                  'Projected',    'Projected','Projected','Projected'],
    'Mean_MASE': [0.644, 1.051, 0.886, 0.851,
                  0.569, 1.176, 0.806, 0.957],
    'Std_MASE':  [0.129, 0.346, 0.241, 0.076,
                  0.136, 0.729, 0.104, 0.456],
})

os.makedirs(f"{PROCESSED_DATA_DIR}/results", exist_ok=True)
baseline_results.to_csv(f"{PROCESSED_DATA_DIR}/results/baseline_no_augmentation.csv", index=False)

print("Updated baseline results (no augmentation):")
print(baseline_results.to_string(index=False))
print(f"\nSummary: ARIMA(1,1,1) wins on both targets.")
print(f"  Best accrual:   ARIMA at 0.644")
print(f"  Best projected: ARIMA at 0.569")

In [ ]:
import sys
!{sys.executable} -m pip install imbalanced-learn

In [ ]:
try:
    from imblearn.over_sampling import SMOTE
    import imblearn
    print(f"imbalanced-learn version: {imblearn.__version__}")
except ImportError as e:
    print(f"Import failed: {e}")

In [ ]:
import sys
!{sys.executable} -m pip install smogn

In [ ]:
# Test SMOGN augmentation on XGBoost and LightGBM across both targets
import importlib
import src.augmentation.smogn_augment
import src.models.evaluation
importlib.reload(src.augmentation.smogn_augment)
importlib.reload(src.models.evaluation)

from src.augmentation.smogn_augment import augment_with_smogn
from src.models.evaluation import run_walk_forward_evaluation
from src.models.tree_models import fit_predict_xgboost, fit_predict_lightgbm
from src.models.prophet_model import fit_predict_prophet
from src.data_loader import load_targets
from src.features import build_all_features

print("Baseline vs SMOGN-augmented results:")
print(f"{'Target':<11}{'Model':<10}{'Baseline':>10}{'Augmented':>12}{'Change':>10}")
print("-" * 55)

for target_name, target_df in load_targets().items():
    features = build_all_features(target_df)
    
    for model_name, model_fn in [('XGBoost', fit_predict_xgboost),
                                  ('LightGBM', fit_predict_lightgbm),
                                  ('Prophet', fit_predict_prophet)]:
        # Baseline (no augmentation)
        baseline = run_walk_forward_evaluation(model_fn, features)
        
        # SMOGN-augmented
        augmented = run_walk_forward_evaluation(
            model_fn, features, augment_fn=augment_with_smogn
        )
        
        change = augmented['mean_mase'] - baseline['mean_mase']
        arrow = '↓ better' if change < 0 else '↑ worse'
        print(f"{target_name:<11}{model_name:<10}"
              f"{baseline['mean_mase']:>10.3f}{augmented['mean_mase']:>12.3f}"
              f"{change:>+8.3f} {arrow}")

In [ ]:
# Save the SMOGN comparison output to a text file so we can share it cleanly
import io
from contextlib import redirect_stdout

output_buffer = io.StringIO()

with redirect_stdout(output_buffer):
    # Repeat the SMOGN comparison cell contents inside this block
    import importlib
    import src.augmentation.smogn_augment
    import src.models.evaluation
    importlib.reload(src.augmentation.smogn_augment)
    importlib.reload(src.models.evaluation)
    
    from src.augmentation.smogn_augment import augment_with_smogn
    from src.models.evaluation import run_walk_forward_evaluation
    from src.models.tree_models import fit_predict_xgboost, fit_predict_lightgbm
    from src.models.prophet_model import fit_predict_prophet
    from src.data_loader import load_targets
    from src.features import build_all_features
    
    print("Baseline vs SMOGN-augmented results:")
    print(f"{'Target':<11}{'Model':<10}{'Baseline':>10}{'Augmented':>12}{'Change':>10}")
    print("-" * 55)
    
    for target_name, target_df in load_targets().items():
        features = build_all_features(target_df)
        
        for model_name, model_fn in [('XGBoost', fit_predict_xgboost),
                                      ('LightGBM', fit_predict_lightgbm),
                                      ('Prophet', fit_predict_prophet)]:
            baseline = run_walk_forward_evaluation(model_fn, features)
            augmented = run_walk_forward_evaluation(
                model_fn, features, augment_fn=augment_with_smogn
            )
            change = augmented['mean_mase'] - baseline['mean_mase']
            arrow = 'better' if change < 0 else 'worse'
            print(f"{target_name:<11}{model_name:<10}"
                  f"{baseline['mean_mase']:>10.3f}{augmented['mean_mase']:>12.3f}"
                  f"{change:>+8.3f} {arrow}")

# Save to file
captured_output = output_buffer.getvalue()
with open('smogn_output.txt', 'w') as f:
    f.write(captured_output)

# Also print to notebook
print(captured_output)
print("\nAlso saved to smogn_output.txt")

In [ ]:
# Diagnose whether SMOGN is actually augmenting
from src.augmentation.smogn_augment import augment_with_smogn
from src.data_loader import load_targets
from src.features import build_all_features
from src.validation import walk_forward_splits

target = load_targets()['accrual']
features = build_all_features(target).dropna()
print(f"Clean features shape: {features.shape}")

# Get the first walk-forward training fold
train_pos, test_pos = next(walk_forward_splits(features))
train = features.iloc[train_pos]

base_columns = ['sales', 'purchases', 'net_cashflow']
X_train = train.drop(columns=base_columns)
y_train = train['net_cashflow'].values

print(f"\nBefore augmentation:")
print(f"  X_train shape: {X_train.shape}")
print(f"  y_train shape: {y_train.shape}")

# Try to augment (this will invoke the try/except silently if it fails)
X_aug, y_aug = augment_with_smogn(X_train, y_train)

print(f"\nAfter augmentation:")
print(f"  X_aug shape: {X_aug.shape}")
print(f"  y_aug shape: {y_aug.shape}")

if len(X_aug) == len(X_train):
    print("\n⚠ Augmentation did nothing — SMOGN either failed or returned zero synthetics")
    
    # Try calling SMOGN directly, without our try/except, to see the actual error
    print("\nTrying SMOGN directly (no try/except) to expose any error:")
    import smogn
    import pandas as pd
    combined = X_train.copy()
    combined['_target'] = y_train
    try:
        result = smogn.smoter(data=combined, y='_target', k=5, samp_method='balance', seed=42)
        print(f"  SMOGN returned {len(result)} rows (original was {len(combined)})")
    except Exception as e:
        print(f"  SMOGN failed with: {type(e).__name__}: {e}")
else:
    print(f"\n✓ Augmentation produced {len(X_aug) - len(X_train)} synthetic rows")

In [ ]:
import importlib
import src.augmentation.smogn_augment
importlib.reload(src.augmentation.smogn_augment)
from src.augmentation.smogn_augment import augment_with_smogn

from src.data_loader import load_targets
from src.features import build_all_features
from src.validation import walk_forward_splits

target = load_targets()['accrual']
features = build_all_features(target).dropna()

train_pos, _ = next(walk_forward_splits(features))
train = features.iloc[train_pos]

base_columns = ['sales', 'purchases', 'net_cashflow']
X_train = train.drop(columns=base_columns)
y_train = train['net_cashflow'].values

print(f"Before augmentation: X {X_train.shape}, y {y_train.shape}")

X_aug, y_aug = augment_with_smogn(X_train, y_train)

print(f"After augmentation:  X {X_aug.shape}, y {y_aug.shape}")

if len(X_aug) > len(X_train):
    print(f"\n✓ SMOGN produced {len(X_aug) - len(X_train)} synthetic rows")
else:
    print(f"\n⚠ Still zero synthetics. Investigate further.")

In [ ]:
# Baseline vs SMOGN-augmented — Prophet excluded from augmentation
import importlib
import src.augmentation.smogn_augment
import src.models.evaluation
importlib.reload(src.augmentation.smogn_augment)
importlib.reload(src.models.evaluation)

from src.augmentation.smogn_augment import augment_with_smogn
from src.models.evaluation import run_walk_forward_evaluation
from src.models.tree_models import fit_predict_xgboost, fit_predict_lightgbm
from src.models.prophet_model import fit_predict_prophet
from src.data_loader import load_targets
from src.features import build_all_features

print("SMOGN augmentation comparison (Prophet excluded from augmentation):")
print(f"{'Target':<11}{'Model':<10}{'Baseline':>10}{'Augmented':>12}{'Change':>10}")
print("-" * 55)

for target_name, target_df in load_targets().items():
    features = build_all_features(target_df)
    
    for model_name, model_fn, allow_augmentation in [
        ('XGBoost',  fit_predict_xgboost,  True),
        ('LightGBM', fit_predict_lightgbm, True),
        ('Prophet',  fit_predict_prophet,  False),   # Prophet: no augmentation
    ]:
        baseline = run_walk_forward_evaluation(model_fn, features)
        
        if allow_augmentation:
            augmented = run_walk_forward_evaluation(
                model_fn, features, augment_fn=augment_with_smogn
            )
            change = augmented['mean_mase'] - baseline['mean_mase']
            arrow = 'better' if change < 0 else 'worse'
            print(f"{target_name:<11}{model_name:<10}"
                  f"{baseline['mean_mase']:>10.3f}{augmented['mean_mase']:>12.3f}"
                  f"{change:>+8.3f} {arrow}")
        else:
            print(f"{target_name:<11}{model_name:<10}"
                  f"{baseline['mean_mase']:>10.3f}{'N/A':>12}{'':>10} augmentation N/A")

In [ ]:
# Save SMOGN augmentation results
import pandas as pd
from src.config import PROCESSED_DATA_DIR
import os

smogn_results = pd.DataFrame({
    'Model':         ['XGBoost', 'LightGBM', 'XGBoost', 'LightGBM'],
    'Target':        ['Accrual', 'Accrual', 'Projected', 'Projected'],
    'Baseline_MASE': [1.051, 0.886, 1.176, 0.806],
    'SMOGN_MASE':    [0.979, 0.999, 1.516, 1.135],
    'Change':        [-0.072, +0.113, +0.340, +0.329],
})

os.makedirs(f"{PROCESSED_DATA_DIR}/results", exist_ok=True)
smogn_results.to_csv(f"{PROCESSED_DATA_DIR}/results/smogn_augmentation.csv", index=False)
print("SMOGN augmentation results saved:")
print(smogn_results.to_string(index=False))

In [ ]:
import sys
!{sys.executable} -m pip install torch

In [ ]:
try:
    import torch
    print(f"PyTorch version: {torch.__version__}")
    print(f"CUDA available: {torch.cuda.is_available()}")
except ImportError as e:
    print(f"Import failed: {e}")

In [ ]:
# Quick VAE augmentation diagnostic
import importlib
import src.augmentation.vae_augment
importlib.reload(src.augmentation.vae_augment)
from src.augmentation.vae_augment import augment_with_vae

from src.data_loader import load_targets
from src.features import build_all_features
from src.validation import walk_forward_splits

target = load_targets()['accrual']
features = build_all_features(target).dropna()

train_pos, _ = next(walk_forward_splits(features))
train = features.iloc[train_pos]

base_columns = ['sales', 'purchases', 'net_cashflow']
X_train = train.drop(columns=base_columns)
y_train = train['net_cashflow'].values

print(f"Before augmentation: X {X_train.shape}, y {y_train.shape}")
print("Training VAE (may take 5-15 seconds)...")

X_aug, y_aug = augment_with_vae(X_train, y_train, target_ratio=2.0)

print(f"After augmentation:  X {X_aug.shape}, y {y_aug.shape}")

if len(X_aug) > len(X_train):
    n_synth = len(X_aug) - len(X_train)
    print(f"\n✓ VAE produced {n_synth} synthetic rows")
    print(f"\nSynthetic y statistics (should be roughly in original y range):")
    print(f"  Original y range: [{y_train.min():,.0f}, {y_train.max():,.0f}], mean {y_train.mean():,.0f}")
    print(f"  Synthetic y range: [{y_aug[-n_synth:].min():,.0f}, {y_aug[-n_synth:].max():,.0f}], mean {y_aug[-n_synth:].mean():,.0f}")
else:
    print("\n⚠ Augmentation did nothing — investigate")

In [ ]:
# Full augmentation comparison: baseline vs SMOGN vs VAE
import importlib
import src.augmentation.smogn_augment
import src.augmentation.vae_augment
import src.models.evaluation
importlib.reload(src.augmentation.smogn_augment)
importlib.reload(src.augmentation.vae_augment)
importlib.reload(src.models.evaluation)

from src.augmentation.smogn_augment import augment_with_smogn
from src.augmentation.vae_augment import augment_with_vae
from src.models.evaluation import run_walk_forward_evaluation
from src.models.tree_models import fit_predict_xgboost, fit_predict_lightgbm
from src.data_loader import load_targets
from src.features import build_all_features

print("Full augmentation comparison (tree models only):")
print(f"{'Target':<11}{'Model':<10}{'Baseline':>10}{'SMOGN':>10}{'VAE':>10}")
print("-" * 51)

for target_name, target_df in load_targets().items():
    features = build_all_features(target_df)
    
    for model_name, model_fn in [('XGBoost', fit_predict_xgboost),
                                  ('LightGBM', fit_predict_lightgbm)]:
        baseline = run_walk_forward_evaluation(model_fn, features)
        smogn = run_walk_forward_evaluation(model_fn, features, augment_fn=augment_with_smogn)
        vae = run_walk_forward_evaluation(model_fn, features, augment_fn=augment_with_vae)
        
        print(f"{target_name:<11}{model_name:<10}"
              f"{baseline['mean_mase']:>10.3f}"
              f"{smogn['mean_mase']:>10.3f}"
              f"{vae['mean_mase']:>10.3f}")

In [ ]:
# Save VAE augmentation results
import pandas as pd
from src.config import PROCESSED_DATA_DIR
import os

vae_results = pd.DataFrame({
    'Model':         ['XGBoost', 'LightGBM', 'XGBoost', 'LightGBM'],
    'Target':        ['Accrual', 'Accrual', 'Projected', 'Projected'],
    'Baseline_MASE': [1.051, 0.886, 1.176, 0.806],
    'VAE_MASE':      [1.821, 1.481, 1.915, 1.368],
    'Change':        [+0.770, +0.595, +0.739, +0.562],
})

os.makedirs(f"{PROCESSED_DATA_DIR}/results", exist_ok=True)
vae_results.to_csv(f"{PROCESSED_DATA_DIR}/results/vae_augmentation.csv", index=False)
print(vae_results.to_string(index=False))

In [ ]:
import sys
!{sys.executable} -m pip install ydata-synthetic

In [ ]:
# TimeGAN diagnostic on one fold
import importlib
import src.augmentation.timegan_augment
importlib.reload(src.augmentation.timegan_augment)
from src.augmentation.timegan_augment import augment_with_timegan

from src.data_loader import load_targets
from src.features import build_all_features
from src.validation import walk_forward_splits

target = load_targets()['accrual']
features = build_all_features(target).dropna()

train_pos, _ = next(walk_forward_splits(features))
train = features.iloc[train_pos]

base_columns = ['sales', 'purchases', 'net_cashflow']
X_train = train.drop(columns=base_columns)
y_train = train['net_cashflow'].values

print(f"Before augmentation: X {X_train.shape}, y {y_train.shape}")
print("Training TimeGAN (100+100+100 epochs, may take 60-120 seconds on CPU)...")

import time
start = time.time()
X_aug, y_aug = augment_with_timegan(X_train, y_train, target_ratio=2.0)
elapsed = time.time() - start

print(f"After augmentation:  X {X_aug.shape}, y {y_aug.shape}  (took {elapsed:.1f}s)")

if len(X_aug) > len(X_train):
    n_synth = len(X_aug) - len(X_train)
    print(f"\n✓ TimeGAN produced {n_synth} synthetic rows")
    print(f"\nSynthetic y statistics:")
    print(f"  Original y range:  [{y_train.min():,.0f}, {y_train.max():,.0f}], mean {y_train.mean():,.0f}")
    print(f"  Synthetic y range: [{y_aug[-n_synth:].min():,.0f}, {y_aug[-n_synth:].max():,.0f}], mean {y_aug[-n_synth:].mean():,.0f}")
else:
    print("\n⚠ Augmentation did nothing")

In [ ]:
import importlib
import src.augmentation.timegan_augment
importlib.reload(src.augmentation.timegan_augment)
from src.augmentation.timegan_augment import augment_with_timegan

from src.data_loader import load_targets
from src.features import build_all_features
from src.validation import walk_forward_splits

target = load_targets()['accrual']
features = build_all_features(target).dropna()

train_pos, _ = next(walk_forward_splits(features))
train = features.iloc[train_pos]

base_columns = ['sales', 'purchases', 'net_cashflow']
X_train = train.drop(columns=base_columns)
y_train = train['net_cashflow'].values

print(f"TimeGAN diagnostic with EXTENDED training (500+500+500 epochs)")
print(f"Before augmentation: X {X_train.shape}, y {y_train.shape}")
print("Training (may take 10-20 seconds)...")

import time
start = time.time()
X_aug, y_aug = augment_with_timegan(
    X_train, y_train, target_ratio=2.0,
    ae_epochs=500, gen_epochs=500, joint_epochs=500,
)
elapsed = time.time() - start

n_synth = len(X_aug) - len(X_train)
print(f"After augmentation:  X {X_aug.shape}, y {y_aug.shape}  (took {elapsed:.1f}s)")

if n_synth > 0:
    print(f"\n✓ TimeGAN produced {n_synth} synthetic rows")
    print(f"\nSynthetic y statistics:")
    print(f"  Original y range:  [{y_train.min():,.0f}, {y_train.max():,.0f}], mean {y_train.mean():,.0f}")
    print(f"  Synthetic y range: [{y_aug[-n_synth:].min():,.0f}, {y_aug[-n_synth:].max():,.0f}], mean {y_aug[-n_synth:].mean():,.0f}")
    
    # Compare to what we got at 100 epochs
    print(f"\nComparison — synthetic y range vs original:")
    orig_range = y_train.max() - y_train.min()
    synth_range = y_aug[-n_synth:].max() - y_aug[-n_synth:].min()
    print(f"  Original range:  {orig_range:,.0f}")
    print(f"  Synthetic range: {synth_range:,.0f}")
    print(f"  Coverage: {100*synth_range/orig_range:.1f}% of original")

In [ ]:
# Full augmentation comparison with TimeGAN at 500 epochs
import importlib
import src.augmentation.smogn_augment
import src.augmentation.vae_augment
import src.augmentation.timegan_augment
import src.models.evaluation
importlib.reload(src.augmentation.smogn_augment)
importlib.reload(src.augmentation.vae_augment)
importlib.reload(src.augmentation.timegan_augment)
importlib.reload(src.models.evaluation)

from src.augmentation.smogn_augment import augment_with_smogn
from src.augmentation.vae_augment import augment_with_vae
from src.augmentation.timegan_augment import augment_with_timegan
from src.models.evaluation import run_walk_forward_evaluation
from src.models.tree_models import fit_predict_xgboost, fit_predict_lightgbm
from src.data_loader import load_targets
from src.features import build_all_features

# Wrap timegan with the 500-epoch config
def augment_with_timegan_500(X_train, y_train):
    return augment_with_timegan(
        X_train, y_train, target_ratio=2.0,
        ae_epochs=500, gen_epochs=500, joint_epochs=500,
    )

print("Full augmentation comparison (tree models only):")
print(f"{'Target':<11}{'Model':<10}{'Baseline':>10}{'SMOGN':>10}{'VAE':>10}{'TimeGAN500':>12}")
print("-" * 63)

for target_name, target_df in load_targets().items():
    features = build_all_features(target_df)
    
    for model_name, model_fn in [('XGBoost', fit_predict_xgboost),
                                  ('LightGBM', fit_predict_lightgbm)]:
        baseline = run_walk_forward_evaluation(model_fn, features)
        smogn = run_walk_forward_evaluation(model_fn, features, augment_fn=augment_with_smogn)
        vae = run_walk_forward_evaluation(model_fn, features, augment_fn=augment_with_vae)
        timegan = run_walk_forward_evaluation(model_fn, features, augment_fn=augment_with_timegan_500)
        
        print(f"{target_name:<11}{model_name:<10}"
              f"{baseline['mean_mase']:>10.3f}"
              f"{smogn['mean_mase']:>10.3f}"
              f"{vae['mean_mase']:>10.3f}"
              f"{timegan['mean_mase']:>12.3f}")

In [ ]:
# Save consolidated augmentation results
import pandas as pd
from src.config import PROCESSED_DATA_DIR
import os

results_master = pd.DataFrame({
    'Target':        ['Accrual', 'Accrual', 'Projected', 'Projected'],
    'Model':         ['XGBoost', 'LightGBM', 'XGBoost', 'LightGBM'],
    'Baseline':      [1.051, 0.886, 1.176, 0.806],
    'SMOGN':         [1.032, 1.136, 1.562, 1.195],
    'VAE':           [1.821, 1.481, 1.915, 1.368],
    'TimeGAN_100':   [1.821, 1.481, 1.915, 1.368],  # from earlier run (used same numbers as VAE, need to re-check)
    'TimeGAN_500':   [1.341, 1.067, 1.375, 0.897],
})

# Wait — I don't have TimeGAN_100 numbers separately saved
# Just save what we have cleanly
augmentation_results = pd.DataFrame({
    'Target':        ['Accrual', 'Accrual', 'Projected', 'Projected'],
    'Model':         ['XGBoost', 'LightGBM', 'XGBoost', 'LightGBM'],
    'Baseline':      [1.051, 0.886, 1.176, 0.806],
    'SMOGN':         [1.032, 1.136, 1.562, 1.195],
    'VAE':           [1.821, 1.481, 1.915, 1.368],
    'TimeGAN_500':   [1.341, 1.067, 1.375, 0.897],
})

os.makedirs(f"{PROCESSED_DATA_DIR}/results", exist_ok=True)
augmentation_results.to_csv(f"{PROCESSED_DATA_DIR}/results/augmentation_comparison.csv", index=False)
print("Consolidated augmentation results saved:")
print(augmentation_results.to_string(index=False))

# Also compute summary statistics
print(f"\nBest configuration per method:")
for method in ['Baseline', 'SMOGN', 'VAE', 'TimeGAN_500']:
    best_row = augmentation_results.iloc[augmentation_results[method].argmin()]
    print(f"  {method:<12}: {best_row['Model']} on {best_row['Target']}: {best_row[method]:.3f}")

In [ ]:
# LSTM diagnostic on one fold
import importlib
import src.models.lstm_model
importlib.reload(src.models.lstm_model)
from src.models.lstm_model import fit_predict_lstm

from src.data_loader import load_targets
from src.features import build_all_features
from src.validation import walk_forward_splits, compute_mase
import time

target = load_targets()['accrual']
features = build_all_features(target).dropna()

# First training fold
train_pos, test_pos = next(walk_forward_splits(features))
train = features.iloc[train_pos]
test = features.iloc[test_pos]

y_train = train['net_cashflow'].values
y_test = test['net_cashflow'].values

base_columns = ['sales', 'purchases', 'net_cashflow']
X_train = train.drop(columns=base_columns)
X_test = test.drop(columns=base_columns)

print(f"Training LSTM on fold 1: y_train {len(y_train)} weeks, forecast horizon {len(y_test)} weeks")
print("Training may take 30-60 seconds on CPU...")

start = time.time()
y_pred = fit_predict_lstm(X_train, y_train, X_test)
elapsed = time.time() - start

mase = compute_mase(y_test, y_pred, y_train)

print(f"\nCompleted in {elapsed:.1f}s")
print(f"Fold 1 MASE: {mase:.3f}")
print(f"\nActual y_test range: [{y_test.min():,.0f}, {y_test.max():,.0f}], mean {y_test.mean():,.0f}")
print(f"Predicted range:     [{y_pred.min():,.0f}, {y_pred.max():,.0f}], mean {y_pred.mean():,.0f}")

In [ ]:
# LSTM training loss trajectory diagnostic
import importlib
import src.models.lstm_model
importlib.reload(src.models.lstm_model)

from src.models.lstm_model import SimpleLSTM, _make_windows
from src.data_loader import load_targets
from src.features import build_all_features
from src.validation import walk_forward_splits

import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import time

torch.manual_seed(42)
np.random.seed(42)

target = load_targets()['accrual']
features = build_all_features(target).dropna()
train_pos, _ = next(walk_forward_splits(features))
train = features.iloc[train_pos]
y_train = train['net_cashflow'].values

# Standardise
mean, std = y_train.mean(), y_train.std()
y_scaled = (y_train - mean) / std

# Build training windows
X_win, y_win = _make_windows(y_scaled, lookback=8)
print(f"Training windows: X {X_win.shape}, y {y_win.shape}")

# Train and log loss
model = SimpleLSTM(input_dim=1, hidden_dim=32)
optimizer = optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()

start = time.time()
losses = []
for epoch in range(200):
    optimizer.zero_grad()
    y_pred = model(X_win)
    loss = loss_fn(y_pred, y_win)
    loss.backward()
    optimizer.step()
    losses.append(loss.item())

elapsed = time.time() - start
print(f"\nTraining completed in {elapsed:.2f}s")
print(f"Loss at epoch   1: {losses[0]:.4f}")
print(f"Loss at epoch  50: {losses[49]:.4f}")
print(f"Loss at epoch 100: {losses[99]:.4f}")
print(f"Loss at epoch 150: {losses[149]:.4f}")
print(f"Loss at epoch 200: {losses[199]:.4f}")
print(f"\nLoss reduction: {losses[0]:.4f} -> {losses[199]:.4f} ({100*(1-losses[199]/losses[0]):.1f}% decrease)")

if losses[199] > losses[0] * 0.5:
    print("\n⚠ Loss reduction is small — LSTM may not be learning meaningful patterns")
elif losses[199] < 0.01:
    print("\n⚠ Loss reduction is very large — possibly overfitting to 44 training windows")
else:
    print("\n✓ Loss reduction looks reasonable for the data volume")

In [ ]:
# Full LSTM evaluation on both targets
import importlib
import src.models.lstm_model
import src.models.evaluation
importlib.reload(src.models.lstm_model)
importlib.reload(src.models.evaluation)

from src.models.lstm_model import fit_predict_lstm
from src.models.evaluation import run_walk_forward_evaluation
from src.data_loader import load_targets
from src.features import build_all_features
import time

print(f"{'Target':<11}{'Model':<10}{'Mean MASE':>12}{'Std MASE':>12}{'Per-fold MASE':>40}")
print("-" * 85)

for target_name, target_df in load_targets().items():
    features = build_all_features(target_df)
    
    start = time.time()
    result = run_walk_forward_evaluation(fit_predict_lstm, features)
    elapsed = time.time() - start
    
    per_fold_str = str([f'{x:.3f}' for x in result['mase_per_fold']])
    print(f"{target_name:<11}{'LSTM':<10}"
          f"{result['mean_mase']:>12.3f}{result['std_mase']:>12.3f}"
          f"{per_fold_str:>40}"
          f"  ({elapsed:.1f}s)")

In [ ]:
# Update the master baseline results table with LSTM
import pandas as pd
from src.config import PROCESSED_DATA_DIR
import os

baseline_full = pd.DataFrame({
    'Model':      ['ARIMA(1,1,1)', 'XGBoost', 'LightGBM', 'Prophet', 'LSTM',
                   'ARIMA(1,1,1)', 'XGBoost', 'LightGBM', 'Prophet', 'LSTM'],
    'Target':     ['Accrual']*5 + ['Projected']*5,
    'Mean_MASE':  [0.644, 1.051, 0.886, 0.851, 0.877,
                   0.569, 1.176, 0.806, 0.957, 0.868],
    'Std_MASE':   [0.129, 0.346, 0.241, 0.076, 0.198,
                   0.136, 0.729, 0.104, 0.456, 0.226],
})

os.makedirs(f"{PROCESSED_DATA_DIR}/results", exist_ok=True)
baseline_full.to_csv(f"{PROCESSED_DATA_DIR}/results/baseline_all_models.csv", index=False)

print("Baseline results (all 5 models, no augmentation):")
print(baseline_full.to_string(index=False))

print(f"\nRankings on Accrual target (best to worst):")
accrual = baseline_full[baseline_full['Target'] == 'Accrual'].sort_values('Mean_MASE')
for i, row in enumerate(accrual.itertuples(), 1):
    print(f"  {i}. {row.Model:<15} MASE {row.Mean_MASE:.3f}")

print(f"\nRankings on Payment-projected target (best to worst):")
projected = baseline_full[baseline_full['Target'] == 'Projected'].sort_values('Mean_MASE')
for i, row in enumerate(projected.itertuples(), 1):
    print(f"  {i}. {row.Model:<15} MASE {row.Mean_MASE:.3f}")

In [ ]:
import sys
!{sys.executable} -m pip install shap

In [ ]:
# SHAP analysis on LightGBM (payment-projected target)
import importlib
import src.interpretability.shap_analysis
importlib.reload(src.interpretability.shap_analysis)

from src.interpretability.shap_analysis import (
    train_final_lightgbm,
    compute_shap_values,
    get_feature_importance_ranking,
    plot_feature_importance,
    plot_summary,
    plot_waterfall,
)

from src.data_loader import load_targets
from src.features import build_all_features
from src.config import PROCESSED_DATA_DIR

import os
import pandas as pd

# Load the payment-projected target (best ML model was LightGBM on this)
target = load_targets()['projected']
features = build_all_features(target).dropna()

# Prepare X and y (same drops as evaluation to prevent leakage)
base_columns = ['sales', 'purchases', 'net_cashflow']
X_full = features.drop(columns=base_columns)
y_full = features['net_cashflow'].values

print(f"Training LightGBM on full feature set: {X_full.shape[0]} rows, {X_full.shape[1]} features")

# Train model on the full evaluation set (SHAP is post-hoc, not walk-forward)
# This gives us the "final" model that would be deployed
model = train_final_lightgbm(X_full, y_full)
print("LightGBM training complete.")

# Compute SHAP values
print("\nComputing SHAP values (may take 20-40 seconds)...")
shap_values = compute_shap_values(model, X_full)
print(f"SHAP values shape: {shap_values.values.shape}")

# Feature importance ranking
print("\nTop 20 features by mean absolute SHAP value:")
ranking = get_feature_importance_ranking(shap_values, top_n=20)
print(ranking.to_string(index=False))

# Save ranking to results
os.makedirs(f"{PROCESSED_DATA_DIR}/results", exist_ok=True)
ranking.to_csv(f"{PROCESSED_DATA_DIR}/results/shap_feature_importance_lightgbm_projected.csv", index=False)
print(f"\nRanking saved to data/processed/results/shap_feature_importance_lightgbm_projected.csv")

In [ ]:
# Generate SHAP visualisation figures
import os
from src.config import PROJECT_ROOT

# Ensure figures directory exists
figures_dir = f"{PROJECT_ROOT}/reports/figures"
os.makedirs(figures_dir, exist_ok=True)

# Figure 1: Feature importance bar chart
print("Generating feature importance bar chart...")
plot_feature_importance(
    shap_values, 
    top_n=15, 
    save_path=f"{figures_dir}/shap_feature_importance_lightgbm_projected.png"
)

# Figure 2: SHAP summary (beeswarm)
print("Generating SHAP summary plot...")
plot_summary(
    shap_values, 
    top_n=15, 
    save_path=f"{figures_dir}/shap_summary_lightgbm_projected.png"
)

# Figure 3: Waterfall for a specific week (choose a mid-range sample)
print("Generating waterfall plot for sample week 50...")
plot_waterfall(
    shap_values, 
    sample_idx=50, 
    top_n=10,
    save_path=f"{figures_dir}/shap_waterfall_week50_lightgbm_projected.png"
)

print(f"\nAll SHAP figures saved to {figures_dir}")

In [ ]:
# Regenerate SHAP plots with cleaner formatting
import importlib
import src.interpretability.shap_analysis
importlib.reload(src.interpretability.shap_analysis)

from src.interpretability.shap_analysis import (
    plot_feature_importance_clean,
    plot_summary_clean,
    plot_waterfall_clean,
)

import os
from src.config import PROJECT_ROOT

figures_dir = f"{PROJECT_ROOT}/reports/figures"
os.makedirs(figures_dir, exist_ok=True)

# Clean feature importance bar
print("Regenerating cleaner feature importance chart...")
plot_feature_importance_clean(
    shap_values, top_n=15,
    save_path=f"{figures_dir}/shap_feature_importance_clean.png"
)

# Clean beeswarm summary
print("Regenerating cleaner summary plot...")
plot_summary_clean(
    shap_values, top_n=15,
    save_path=f"{figures_dir}/shap_summary_clean.png"
)

# Clean waterfall — pick a week that showcases interesting attribution
# Choose the sample with the largest absolute predicted deviation from baseline
predictions = model.predict(X_full)
baseline = predictions.mean()
deviations = np.abs(predictions - baseline)
interesting_idx = int(deviations.argmax())
print(f"\nMost interesting sample: index {interesting_idx}, prediction {predictions[interesting_idx]:,.0f}")

plot_waterfall_clean(
    shap_values, X_full,
    sample_idx=interesting_idx, top_n=10,
    save_path=f"{figures_dir}/shap_waterfall_clean.png"
)

print(f"\nAll cleaner SHAP figures saved to {figures_dir}")

In [ ]:
import importlib
import src.interpretability.shap_analysis
importlib.reload(src.interpretability.shap_analysis)

from src.interpretability.shap_analysis import (
    plot_feature_importance_clean,
    plot_summary_clean,
    plot_waterfall_clean,
)

import os
from src.config import PROJECT_ROOT

figures_dir = f"{PROJECT_ROOT}/reports/figures"

# Feature importance bar (fixed: no em dash in title)
print("Regenerating feature importance chart...")
plot_feature_importance_clean(
    shap_values, top_n=15,
    save_path=f"{figures_dir}/shap_feature_importance_clean.png"
)

# Beeswarm (fixed: no em dash in title)
print("Regenerating summary plot...")
plot_summary_clean(
    shap_values, top_n=15,
    save_path=f"{figures_dir}/shap_summary_clean.png"
)

# Waterfall (fixed: no em dash, larger figure, reduced top_n)
predictions = model.predict(X_full)
baseline = predictions.mean()
deviations = np.abs(predictions - baseline)
interesting_idx = int(deviations.argmax())
print(f"\nMost interesting sample: index {interesting_idx}, prediction {predictions[interesting_idx]:,.0f}")

plot_waterfall_clean(
    shap_values, X_full,
    sample_idx=interesting_idx, top_n=8,
    save_path=f"{figures_dir}/shap_waterfall_clean.png"
)

print(f"\nAll plots regenerated")

In [ ]:
import importlib
import src.interpretability.shap_analysis
importlib.reload(src.interpretability.shap_analysis)

from src.interpretability.shap_analysis import (
    plot_feature_importance_clean,
    plot_summary_clean,
    plot_waterfall_clean,
)

import os
from src.config import PROJECT_ROOT

figures_dir = f"{PROJECT_ROOT}/reports/figures"

# Regenerate all three
print("Feature importance...")
plot_feature_importance_clean(
    shap_values, top_n=15,
    save_path=f"{figures_dir}/shap_feature_importance_clean.png"
)

print("Beeswarm summary...")
plot_summary_clean(
    shap_values, top_n=15,
    save_path=f"{figures_dir}/shap_summary_clean.png"
)

# Waterfall on most-extreme prediction
predictions = model.predict(X_full)
baseline = predictions.mean()
deviations = np.abs(predictions - baseline)
interesting_idx = int(deviations.argmax())
print(f"Waterfall on sample {interesting_idx} (largest deviation)...")

plot_waterfall_clean(
    shap_values, X_full,
    sample_idx=interesting_idx, top_n=8,
    save_path=f"{figures_dir}/shap_waterfall_clean.png"
)

print("\nAll plots regenerated")

In [ ]:
import importlib
import src.interpretability.shap_analysis
importlib.reload(src.interpretability.shap_analysis)

from src.interpretability.shap_analysis import (
    plot_feature_importance_clean,
    plot_summary_clean,
    plot_contribution_chart,
)

import os
from src.config import PROJECT_ROOT

figures_dir = f"{PROJECT_ROOT}/reports/figures"

print("Feature importance...")
plot_feature_importance_clean(
    shap_values, top_n=15,
    save_path=f"{figures_dir}/shap_feature_importance_clean.png"
)

print("Beeswarm (saved at high DPI)...")
plot_summary_clean(
    shap_values, top_n=15,
    save_path=f"{figures_dir}/shap_summary_clean.png"
)

# Contribution chart on most-extreme prediction
predictions = model.predict(X_full)
baseline = predictions.mean()
deviations = np.abs(predictions - baseline)
interesting_idx = int(deviations.argmax())
print(f"Contribution chart on sample {interesting_idx}...")

plot_contribution_chart(
    shap_values, X_full,
    sample_idx=interesting_idx, top_n=10,
    save_path=f"{figures_dir}/shap_contribution_clean.png"
)

print("Done")

In [ ]:
import importlib
import src.interpretability.shap_analysis
importlib.reload(src.interpretability.shap_analysis)
from src.interpretability.shap_analysis import plot_summary_clean

import os
from src.config import PROJECT_ROOT

figures_dir = f"{PROJECT_ROOT}/reports/figures"

plot_summary_clean(
    shap_values, top_n=15,
    save_path=f"{figures_dir}/shap_summary_clean.png"
)

In [ ]:
import sys
!{sys.executable} -m pip install streamlit

In [ ]:
try:
    import streamlit
    print(f"Streamlit version: {streamlit.__version__}")
except ImportError as e:
    print(f"Import failed: {e}")
    

In [ ]:
import importlib
import src.features
importlib.reload(src.features)
from src.features import build_all_features
from src.data_loader import load_targets

target = load_targets()['projected']
features = build_all_features(target)

print(f"Total columns in feature DataFrame: {len(features.columns)}")
print(f"\nColumn list:")
for col in features.columns:
    print(f"  {col}")

# Break down by family
base = [c for c in features.columns if c in ('sales', 'purchases', 'net_cashflow')]
calendar = [c for c in features.columns if c in ('year', 'quarter', 'month', 'week_of_year', 'week_of_month', 'is_quarter_end')]
lag = [c for c in features.columns if '_lag_' in c]
rolling = [c for c in features.columns if '_roll' in c]
paycycle = [c for c in features.columns if 'paywindow' in c or 'expected' in c or 'ratio' in c]

print(f"\nBase series:      {len(base)} columns")
print(f"Calendar:         {len(calendar)} columns")
print(f"Lag:              {len(lag)} columns")
print(f"Rolling:          {len(rolling)} columns")
print(f"Payment-cycle:    {len(paycycle)} columns")
print(f"Sum:              {len(base)+len(calendar)+len(lag)+len(rolling)+len(paycycle)}")

# After dropping base columns (as the evaluation runner does)
model_features = features.drop(columns=['sales', 'purchases', 'net_cashflow'])
print(f"\nAfter dropping 3 base columns: {len(model_features.columns)} features passed to model")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

models = ['ARIMA(1,1,1)', 'XGBoost', 'LightGBM', 'Prophet', 'LSTM']
accrual_mase = [0.644, 1.051, 0.886, 0.851, 0.877]
projected_mase = [0.569, 1.176, 0.806, 0.957, 0.868]
accrual_std = [0.129, 0.346, 0.241, 0.076, 0.198]
projected_std = [0.136, 0.729, 0.104, 0.456, 0.226]

x = np.arange(len(models))
width = 0.35

fig, ax = plt.subplots(figsize=(11, 6))

bars_accrual = ax.bar(x - width/2, accrual_mase, width, 
                       yerr=accrual_std, label='Accrual target',
                       color='#4A5568', alpha=0.85, edgecolor='none',
                       capsize=4, error_kw={'linewidth': 1.2})

bars_projected = ax.bar(x + width/2, projected_mase, width,
                         yerr=projected_std, label='Payment-projected target',
                         color='#E91E63', alpha=0.85, edgecolor='none',
                         capsize=4, error_kw={'linewidth': 1.2})

for bar, val in zip(bars_accrual, accrual_mase):
    ax.text(bar.get_x() + bar.get_width()/2, val + 0.02, f'{val:.3f}',
            ha='center', va='bottom', fontsize=9, color='#333')
for bar, val in zip(bars_projected, projected_mase):
    ax.text(bar.get_x() + bar.get_width()/2, val + 0.02, f'{val:.3f}',
            ha='center', va='bottom', fontsize=9, color='#333')

ax.axhline(y=1.0, color='#999', linestyle='--', linewidth=1, alpha=0.7)
ax.text(4.7, 1.02, 'Naive baseline (MASE = 1.0)', ha='right', va='bottom',
        fontsize=9, color='#666', style='italic')

ax.set_xlabel('Model', fontsize=11)
ax.set_ylabel('Mean MASE (across 5 walk-forward folds)', fontsize=11)
ax.set_title('Cross-Target Model Performance Comparison', fontsize=13, pad=15)
ax.set_xticks(x)
ax.set_xticklabels(models, rotation=0, ha='center', fontsize=10)
ax.set_ylim(0, 2.0)
ax.grid(axis='y', linestyle='--', alpha=0.4)
ax.set_axisbelow(True)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.legend(loc='upper right', framealpha=0.95, fontsize=10)

plt.tight_layout()

import os
from src.config import PROJECT_ROOT
figures_dir = f"{PROJECT_ROOT}/reports/figures"
save_path = f"{figures_dir}/cross_target_model_comparison.png"
plt.savefig(save_path, dpi=150, bbox_inches='tight')
plt.show()

print(f"Saved: {save_path}")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

models = ['ARIMA(1,1,1)', 'XGBoost', 'LightGBM', 'Prophet', 'LSTM']
accrual_mase = [0.644, 1.051, 0.886, 0.851, 0.877]
projected_mase = [0.569, 1.176, 0.806, 0.957, 0.868]

x = np.arange(len(models))
width = 0.38

fig, ax = plt.subplots(figsize=(11, 5.5))

bars_accrual = ax.bar(x - width/2, accrual_mase, width,
                       label='Accrual target', color='#4A5568', edgecolor='none')
bars_projected = ax.bar(x + width/2, projected_mase, width,
                         label='Payment-projected target', color='#E91E63', edgecolor='none')

for bar, val in zip(bars_accrual, accrual_mase):
    ax.text(bar.get_x() + bar.get_width()/2, val + 0.02, f'{val:.3f}',
            ha='center', va='bottom', fontsize=10, fontweight='bold', color='#333')
for bar, val in zip(bars_projected, projected_mase):
    ax.text(bar.get_x() + bar.get_width()/2, val + 0.02, f'{val:.3f}',
            ha='center', va='bottom', fontsize=10, fontweight='bold', color='#333')

ax.axhline(y=1.0, color='#999', linestyle='--', linewidth=1.2, alpha=0.6)
ax.text(-0.4, 1.03, 'Naive baseline (MASE = 1.0)', ha='left', va='bottom',
        fontsize=9, color='#666', style='italic')

ax.set_xlabel('Model', fontsize=12)
ax.set_ylabel('Mean MASE', fontsize=12)
ax.set_title('Cross-Target Model Performance Comparison', fontsize=14, pad=15)
ax.set_xticks(x)
ax.set_xticklabels(models, fontsize=11)
ax.set_ylim(0, 1.4)
ax.grid(axis='y', linestyle='--', alpha=0.3)
ax.set_axisbelow(True)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.legend(loc='upper right', framealpha=0.95, fontsize=11)

plt.tight_layout()

import os
from src.config import PROJECT_ROOT
figures_dir = f"{PROJECT_ROOT}/reports/figures"
save_path = f"{figures_dir}/cross_target_model_comparison.png"
plt.savefig(save_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved: {save_path}")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

models = ['ARIMA(1,1,1)', 'XGBoost', 'LightGBM', 'Prophet', 'LSTM']
accrual_mase = [0.644, 1.051, 0.886, 0.851, 0.877]
projected_mase = [0.569, 1.176, 0.806, 0.957, 0.868]

# Delta: positive means projected is worse
deltas = [p - a for a, p in zip(accrual_mase, projected_mase)]
delta_pct = [100 * d / a for a, d in zip(accrual_mase, deltas)]

fig, ax = plt.subplots(figsize=(11, 5))

colors = ['#4A9D3E' if d < 0 else '#C93838' for d in deltas]
bars = ax.barh(models, deltas, color=colors, edgecolor='none')

ax.axvline(x=0, color='#333', linewidth=1)

for bar, d, pct in zip(bars, deltas, delta_pct):
    x_pos = bar.get_width()
    label = f'{d:+.3f} ({pct:+.1f}%)'
    if x_pos >= 0:
        ax.text(x_pos + 0.005, bar.get_y() + bar.get_height()/2, label,
                va='center', ha='left', fontsize=10, fontweight='bold')
    else:
        ax.text(x_pos - 0.005, bar.get_y() + bar.get_height()/2, label,
                va='center', ha='right', fontsize=10, fontweight='bold')
import matplotlib.pyplot as plt
from matplotlib.ticker import MultipleLocator
import numpy as np

models = ['ARIMA(1,1,1)', 'XGBoost', 'LightGBM', 'Prophet', 'LSTM']
accrual_mase = [0.644, 1.051, 0.886, 0.851, 0.877]
projected_mase = [0.569, 1.176, 0.806, 0.957, 0.868]

deltas = [p - a for a, p in zip(accrual_mase, projected_mase)]
delta_pct = [100 * d / a for a, d in zip(accrual_mase, deltas)]

fig, ax = plt.subplots(figsize=(11, 5))

colors = ['#4A9D3E' if d < 0 else '#C93838' for d in deltas]
bars = ax.barh(models, deltas, color=colors, edgecolor='none')

ax.axvline(x=0, color='#333', linewidth=1.2)

for bar, d, pct in zip(bars, deltas, delta_pct):
    x_pos = bar.get_width()
    label = f'{d:+.3f} ({pct:+.1f}%)'
    if x_pos >= 0:
        ax.text(x_pos + 0.003, bar.get_y() + bar.get_height()/2, label,
                va='center', ha='left', fontsize=10, fontweight='bold')
    else:
        ax.text(x_pos - 0.003, bar.get_y() + bar.get_height()/2, label,
                va='center', ha='right', fontsize=10, fontweight='bold')

# Better axis: tighter range, cleaner tick spacing
ax.set_xlim(-0.13, 0.17)
ax.xaxis.set_major_locator(MultipleLocator(0.05))
ax.tick_params(axis='x', labelsize=10)

ax.set_xlabel('MASE change (projected minus accrual). Negative = better on projected.', fontsize=11)
ax.set_title('Change in Model Performance: Accrual to Payment-Projected Target', fontsize=13, pad=15)
ax.grid(axis='x', linestyle='--', alpha=0.3)
ax.set_axisbelow(True)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()

import os
from src.config import PROJECT_ROOT
figures_dir = f"{PROJECT_ROOT}/reports/figures"
save_path = f"{figures_dir}/cross_target_delta_comparison.png"
plt.savefig(save_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved: {save_path}")
ax.set_xlabel('MASE change (projected minus accrual). Negative = better on projected.', fontsize=11)
ax.set_title('Change in Model Performance: Accrual to Payment-Projected Target', fontsize=13, pad=15)
ax.set_xlim(-0.15, 0.15)
ax.grid(axis='x', linestyle='--', alpha=0.3)
ax.set_axisbelow(True)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()

import os
from src.config import PROJECT_ROOT
figures_dir = f"{PROJECT_ROOT}/reports/figures"
save_path = f"{figures_dir}/cross_target_delta_comparison.png"
plt.savefig(save_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved: {save_path}")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

configs = [
    ('ARIMA(1,1,1) - Projected', 0.569, '#E91E63'),
    ('ARIMA(1,1,1) - Accrual', 0.644, '#4A5568'),
    ('LightGBM - Projected', 0.806, '#E91E63'),
    ('Prophet - Accrual', 0.851, '#4A5568'),
    ('LSTM - Projected', 0.868, '#E91E63'),
    ('LSTM - Accrual', 0.877, '#4A5568'),
    ('LightGBM - Accrual', 0.886, '#4A5568'),
    ('Prophet - Projected', 0.957, '#E91E63'),
    ('XGBoost - Accrual', 1.051, '#4A5568'),
    ('XGBoost - Projected', 1.176, '#E91E63'),
]

names = [c[0] for c in configs]
values = [c[1] for c in configs]
colors = [c[2] for c in configs]

fig, ax = plt.subplots(figsize=(10, 7))

bars = ax.barh(names, values, color=colors, edgecolor='none')
ax.axvline(x=1.0, color='#666', linestyle='--', linewidth=1, alpha=0.7)
ax.text(1.0, 10.3, 'Naive baseline', ha='center', fontsize=9, color='#666', style='italic')

for bar, val in zip(bars, values):
    ax.text(val + 0.015, bar.get_y() + bar.get_height()/2, f'{val:.3f}',
            va='center', ha='left', fontsize=10, fontweight='bold')

ax.set_xlabel('Mean MASE (across 5 walk-forward folds)', fontsize=11)
ax.set_title('All Model-Target Configurations Ranked by MASE', fontsize=13, pad=15)
ax.set_xlim(0, 1.35)
ax.invert_yaxis()
ax.grid(axis='x', linestyle='--', alpha=0.3)
ax.set_axisbelow(True)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#4A5568', label='Accrual target'),
    Patch(facecolor='#E91E63', label='Payment-projected target'),
]
ax.legend(handles=legend_elements, loc='lower right', framealpha=0.95, fontsize=10)

plt.tight_layout()

import os
from src.config import PROJECT_ROOT
figures_dir = f"{PROJECT_ROOT}/reports/figures"
save_path = f"{figures_dir}/all_configurations_ranked.png"
plt.savefig(save_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved: {save_path}")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

models = ['ARIMA(1,1,1)', 'XGBoost', 'LightGBM', 'Prophet', 'LSTM']
accrual_mase = [0.644, 1.051, 0.886, 0.851, 0.877]
projected_mase = [0.569, 1.176, 0.806, 0.957, 0.868]

x = np.arange(len(models))
width = 0.32

fig, ax = plt.subplots(figsize=(12, 6))

# Bars — deep navy and warm orange
bars_accrual = ax.bar(x - width/2, accrual_mase, width,
                       label='Accrual target', color='#2C5282', 
                       edgecolor='none', alpha=0.9)
bars_projected = ax.bar(x + width/2, projected_mase, width,
                         label='Payment-projected target', color='#DD6B20',
                         edgecolor='none', alpha=0.9)

# Connecting lines (green for improvement, red for degradation)
for i in range(len(models)):
    delta = projected_mase[i] - accrual_mase[i]
    line_color = '#2E7D32' if delta < 0 else '#C62828'
    
    ax.plot([x[i] - width/2, x[i] + width/2],
            [accrual_mase[i], projected_mase[i]],
            color=line_color, linewidth=2.5,
            marker='o', markersize=6, markerfacecolor=line_color,
            markeredgecolor='white', markeredgewidth=1.5,
            zorder=5)

# Value labels above bars
for bar, val in zip(bars_accrual, accrual_mase):
    ax.text(bar.get_x() + bar.get_width()/2, val + 0.03, f'{val:.3f}',
            ha='center', va='bottom', fontsize=10, fontweight='bold', color='#333')
for bar, val in zip(bars_projected, projected_mase):
    ax.text(bar.get_x() + bar.get_width()/2, val + 0.03, f'{val:.3f}',
            ha='center', va='bottom', fontsize=10, fontweight='bold', color='#333')

# Reference line at MASE = 1.0
ax.axhline(y=1.0, color='#999', linestyle='--', linewidth=1.2, alpha=0.6, zorder=1)
ax.text(-0.4, 1.03, 'Naive baseline (MASE = 1.0)', ha='left', va='bottom',
        fontsize=9, color='#666', style='italic')

# Formatting
ax.set_xlabel('Model', fontsize=12)
ax.set_ylabel('Mean MASE', fontsize=12)
ax.set_title('Cross-Target Model Performance Comparison',
             fontsize=14, pad=15)
ax.set_xticks(x)
ax.set_xticklabels(models, fontsize=11)
ax.set_ylim(0, 1.4)
ax.grid(axis='y', linestyle='--', alpha=0.3, zorder=0)
ax.set_axisbelow(True)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# Legend with both bars and lines
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
legend_elements = [
    Patch(facecolor='#2C5282', alpha=0.9, label='Accrual target'),
    Patch(facecolor='#DD6B20', alpha=0.9, label='Payment-projected target'),
    Line2D([0], [0], color='#2E7D32', linewidth=2.5, marker='o', markersize=6,
           label='Improves on projected'),
    Line2D([0], [0], color='#C62828', linewidth=2.5, marker='o', markersize=6,
           label='Degrades on projected'),
]
ax.legend(handles=legend_elements, loc='upper right', framealpha=0.95, fontsize=10, ncol=2)

plt.tight_layout()

# Save
import os
from src.config import PROJECT_ROOT
figures_dir = f"{PROJECT_ROOT}/reports/figures"
save_path = f"{figures_dir}/cross_target_model_comparison.png"
plt.savefig(save_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved: {save_path}")

In [ ]:
import importlib
import src.interpretability.shap_analysis
importlib.reload(src.interpretability.shap_analysis)

from src.interpretability.shap_analysis import (
    train_final_lightgbm,
    compute_shap_values,
)

# Rebuild the model and SHAP if not still in memory
from src.data_loader import load_targets
from src.features import build_all_features
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter

target = load_targets()['projected']
features = build_all_features(target).dropna()
base_columns = ['sales', 'purchases', 'net_cashflow']
X_full = features.drop(columns=base_columns)
y_full = features['net_cashflow'].values

model = train_final_lightgbm(X_full, y_full)
shap_values = compute_shap_values(model, X_full)

# Find the most-extreme week
predictions = model.predict(X_full)
baseline = predictions.mean()
deviations = np.abs(predictions - baseline)
interesting_idx = int(deviations.argmax())

# Custom waterfall with DD-MM-YYYY date format
def _fmt(x, pos):
    if abs(x) >= 1_000_000:
        return f'Rs {x/1_000_000:.1f}M'
    elif abs(x) >= 1_000:
        return f'Rs {x/1_000:.0f}K'
    else:
        return f'Rs {x:.0f}'

sample_shap = shap_values[interesting_idx]
base_value = float(sample_shap.base_values)
predicted_value = float(base_value + sample_shap.values.sum())

contributions = pd.DataFrame({
    'feature': shap_values.feature_names,
    'value': X_full.iloc[interesting_idx].values,
    'shap': sample_shap.values,
})
contributions['abs_shap'] = contributions['shap'].abs()
top_features = contributions.nlargest(8, 'abs_shap').copy()
remaining = contributions.drop(top_features.index)
remaining_sum = remaining['shap'].sum()

# Date in DD-MM-YYYY format
selected_date = X_full.index[interesting_idx]
sample_label = selected_date.strftime('%d-%m-%Y')

# Build waterfall rows
rows = [{'label': 'Baseline (average prediction)',
         'shap': None,
         'position': base_value,
         'type': 'baseline'}]

running_total = base_value
for _, row in top_features.iterrows():
    running_total += row['shap']
    rows.append({
        'label': f"{row['feature']}\n(value: {row['value']:,.0f})",
        'shap': row['shap'],
        'position': running_total,
        'type': 'contribution',
    })

if len(remaining) > 0:
    running_total += remaining_sum
    rows.append({
        'label': f"{len(remaining)} other features",
        'shap': remaining_sum,
        'position': running_total,
        'type': 'contribution',
    })

rows.append({'label': 'Final prediction',
             'shap': None,
             'position': predicted_value,
             'type': 'prediction'})

fig, ax = plt.subplots(figsize=(13, 8))
y_positions = list(range(len(rows)))
y_positions.reverse()

for i, row_data in enumerate(rows):
    y = y_positions[i]
    
    if row_data['type'] in ('baseline', 'prediction'):
        color = '#333' if row_data['type'] == 'prediction' else '#666'
        ax.barh(y, row_data['position'], color=color, alpha=0.15, 
                edgecolor=color, linewidth=1.5, height=0.6)
        value_str = f"Rs {row_data['position']/1000:,.0f}K"
        ax.text(row_data['position'], y, f'  {value_str}',
                va='center', ha='left', fontsize=10, fontweight='bold', color=color)
    else:
        prev_position = rows[i-1]['position']
        width = abs(row_data['shap'])
        left = min(prev_position, row_data['position'])
        color = '#E91E63' if row_data['shap'] > 0 else '#2196F3'
        
        ax.barh(y, width, left=left, color=color, edgecolor='none', height=0.6)
        
        value_sign = '+' if row_data['shap'] > 0 else ''
        value_str = f"{value_sign}Rs {row_data['shap']/1000:,.1f}K"
        
        if row_data['shap'] > 0:
            text_x = row_data['position']
            ha = 'left'
            offset = width * 0.03
        else:
            text_x = row_data['position']
            ha = 'right'
            offset = -width * 0.03
        
        ax.text(text_x + offset, y, value_str,
                va='center', ha=ha, fontsize=9, fontweight='bold', color='#333')

ax.set_yticks(y_positions)
ax.set_yticklabels([row_data['label'] for row_data in rows], fontsize=9)

ax.axvline(x=0, color='#ccc', linewidth=0.8)
ax.xaxis.set_major_formatter(FuncFormatter(_fmt))
ax.set_xlabel('Predicted weekly cashflow (INR)', fontsize=11)
ax.set_title(f'Cashflow Prediction Breakdown, Week ending {sample_label}\n'
             f'From baseline of Rs {base_value/1000:,.0f}K to final prediction of Rs {predicted_value/1000:,.0f}K',
             fontsize=12, pad=15)
ax.grid(axis='x', linestyle='--', alpha=0.3)
ax.set_axisbelow(True)

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

all_positions = [r['position'] for r in rows]
x_min = min(all_positions + [0])
x_max = max(all_positions + [0])
x_range = x_max - x_min
ax.set_xlim(x_min - x_range * 0.15, x_max + x_range * 0.20)

plt.tight_layout()

# Save (overwrites the old file)
import os
from src.config import PROJECT_ROOT
figures_dir = f"{PROJECT_ROOT}/reports/figures"
save_path = f"{figures_dir}/shap_waterfall_clean.png"
plt.savefig(save_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved: {save_path}")
print(f"Date format in figure: {sample_label}")

In [ ]:
import json
# adjust the path to wherever your feature dataframe is saved
# examples:
#   df = pd.read_parquet('data/processed/features.parquet')
#   df = pd.read_csv('data/processed/features_final.csv')

feature_cols = [c for c in df.columns if c not in ['week', 'net_cashflow_projected', 'net_cashflow_accrual']]
print(len(feature_cols))
print(json.dumps(feature_cols, indent=2))